# Mamba3Yolo — Kaggle Runbook

Mamba-3 (complex-state, exponential-trapezoidal) SSM mixer inside a YOLO11 detector,
plus intrinsic controllability-Gramian explainability.

**Before running:** Settings → Accelerator: GPU (T4/P100), Internet: ON.

## Attribution — read this before claiming anything

This work is **derivative**, and the parts that are not ours are load-bearing:

- `LSBlock`, `RGBlock` and the ODSS block are **Mamba-YOLO**'s
  (Wang et al., AAAI 2025, [arXiv:2406.05835](https://arxiv.org/abs/2406.05835),
  [HZAI-ZJNU/Mamba-YOLO](https://github.com/HZAI-ZJNU/Mamba-YOLO)) — **AGPL-3.0**.
- The trainer, loss, assigner, NMS and mAP are **Ultralytics** — **AGPL-3.0**.
- Swapping C3k2 for an SSM mixer in a YOLO11 yaml is **not new**: Xray-YOLO-Mamba
  (Zhao et al., *Sci Rep* 15:13171, 2025) already replaces C3k2 with a VSS/SS2D block
  in a YOLOv11-n backbone, at 4.3M params / 10.3 GFLOPs / 95.2 FPS.

This repo is therefore AGPL-3.0 and must ship that notice. **Two things here are new**,
and they are the only things to claim:

1. the SSM *inside* the block is Mamba-3 — complex/rotational state + exponential-trapezoidal
   discretization. Every prior YOLO-Mamba variant (Mamba-YOLO, FER-YOLO-Mamba,
   Xray-YOLO-Mamba) uses VMamba-era S6/SS2D: real diagonal decay, ZOH.
2. intrinsic Gramian saliency with faithfulness metrics. The priors show qualitative
   heatmaps only.

## Honesty rules for this notebook

- Report only numbers you measure here. `official_mamba3_kernels = False` — no released
  kernel; the verified pure-PyTorch core runs.
- The efficiency baseline is **not** YOLO11s. It is Mamba-YOLO-T (5.8M / 13.2G / AP 44.5)
  and Xray-YOLO-Mamba (4.3M / 10.3G). Compare against those or say nothing about efficiency.
- coco128 has train == val. It is a pipeline check. It is never a paper number. The effect
  size you are hunting is ~2-3 mAP points (Xray-YOLO-Mamba: 71.9 → 74.6 over YOLOv11-n);
  128 images cannot resolve that.

## 0. Environment

Run this cell FIRST. `ultralytics==8.3.0` pins `numpy<2`, but Kaggle's image is built for numpy 2.x — a normal install downgrades numpy and ABI-breaks cv2/tifffile/shap. `--no-deps` keeps Kaggle's numpy 2.x; ultralytics 8.3.0 runs fine on it.

If numpy shows 1.26.4 below, do Run → Factory reset first.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # reduce CUDA fragmentation
os.environ["WANDB_MODE"] = "disabled"                                # skip the interactive W&B prompt
import importlib, subprocess, sys

def have(m):
    try:
        importlib.import_module(m); return True
    except Exception:
        return False

if not have("ultralytics"):
    # --no-deps: do NOT let pip downgrade Kaggle's numpy 2.x (that breaks cv2's ABI).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "ultralytics==8.3.0", "ultralytics-thop"], check=False)
import ultralytics, numpy, cv2
print("ultralytics", ultralytics.__version__, "| numpy", numpy.__version__, "| cv2", cv2.__version__)

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
import os
if not os.path.exists('Mamba3Yolo'):
    !git clone -q https://github.com/ShMazumder/Mamba3Yolo.git
os.chdir('Mamba3Yolo')
os.makedirs('src/xai', exist_ok=True); os.makedirs('scripts', exist_ok=True)
print('cwd:', os.getcwd())

## 1. Patch in the verified source files

The GitHub repo may lag behind; these cells write the current, gated code.

**What changed vs. the version that produced the all-NaN run:**

- `mamba3_ref.py` — the chunked scan divided by `cumprod(alpha).clamp(min=1e-6)`. Once
  `alpha = exp(dt·A)` fell below ~0.6, `alpha**32` underflowed fp16, the clamp bit, and
  `U / 1e-6` overflowed → `inf` → `NaN` on every batch, so AMP skipped every step and all
  three Mamba runs reported a *frozen* epoch-1 model. Replaced with the bounded segment-sum
  (SSD) form: every `exp()` argument is ≤ 0. Also: scalar-per-head decay (the actual
  Mamba-2/3 structure `A_t = α_t·R_t`) so the readout contracts and the `(B,L,H,P,N)` state
  trajectory is never built; fp32-forced core; bounded + wrapped rotary phase; fixed
  `token_saliency` padding; vectorised `_reverse_energy`.
- `mamba3_odss.py` — one scan path constructed, not two (the old `self._official_mamba = Mamba3(...)`
  registered as a submodule regardless of the underscore).
- `ultra_mamba3.py` — new `where="backbone"` option; no prior work swaps the neck.
- `validate_core.py` — **new**: the gate that would have caught all of the above.

In [ ]:
%%writefile src/blocks/mamba3_ref.py
"""
Faithful pure-PyTorch reference for the Mamba-3 selective SSM.

Implements the actual Mamba-3 recurrence (complex/rotational state + exponential-
trapezoidal discretization), NOT a gated-MLP stand-in.

Recurrence (SISO, per head h, per channel p):
    alpha_t = exp(dt_t * A)                          # scalar decay per head, A < 0
    beta_t  = (1 - lambda_t) * dt_t * alpha_t        # previous-input (trapezoidal) weight
    gamma_t = lambda_t * dt_t                        # current-input weight
    h_t = alpha_t * h_{t-1}
          + gamma_t * outer(x_t,     RoPE(B_t,     Phi_t))
          + beta_t  * outer(x_{t-1}, RoPE(B_{t-1}, Phi_{t-1}))
    y_t = < h_t , RoPE(C_t, Phi_t) >
Phi_t is the cumulative, data-dependent rotary angle; rotating B at write-time and C
at read-time is the real-valued realisation of the complex state transition
A_t = alpha_t * R(theta_t)  (scalar decay TIMES rotation -- the Mamba-2/3 SSD
structure).  lambda_t = 1  ->  exponential-Euler  ==  Mamba-2 (single input term):
that is the knob for the trapezoidal ablation.  use_rope=False -> real-valued
diagonal SSM: the control for the parity/state-tracking gate.

WHAT CHANGED vs. the first version of this file (and why the detector NaN'd)
--------------------------------------------------------------------------
1. The chunked scan used  intra = Pin * cumsum(U / Pin)  with
   Pin = cumprod(alpha).clamp(min=1e-6).  alpha = exp(dt*A) drops well below 1 after
   a few optimizer steps, so alpha**chunk underflows (in fp16 already for
   alpha < ~0.6), the clamp kicks in, and U / 1e-6 overflows fp16 -> inf -> NaN on
   every batch.  That is exactly what the run log shows: NaN from epoch 2 and a
   *frozen* model (identical val metrics every epoch = every AMP step skipped).
   It is now replaced by the segment-sum ("SSD") form:
       h_t = sum_{s<=t} exp(cs_t - cs_s) U_s ,   cs = cumsum(log alpha)
   Every exp() argument is <= 0, so every factor lives in [0, 1].  No division, no
   clamp, no overflow -- and it is *exactly* the same recurrence (see
   scripts/validate_core.py, which checks it against the O(L) loop).
2. A is now a scalar per head (shape (nheads,)) instead of (nheads, d_state).  That
   is the Mamba-2/3 structure (A_t = alpha_t * R_t: scalar decay, rotation carries
   the state-dependence) and it lets the readout be contracted analytically, so the
   (B, L, H, P, N) state trajectory is never materialised.  Memory drops ~30x and
   the block gets several times faster.
3. The SSM math runs with autocast disabled (fp32).  exp / cumsum / rotation over
   L = 4096..16384 tokens are not fp16-safe; the projections still run in fp16.
4. The rotation rate is bounded (rot_max * tanh) and Phi is wrapped mod 2*pi.
   Unbounded w_proj summed over 16k tokens produced |Phi| >> 1e4, which destroys
   cos/sin precision (and overflows fp16 outright).  pi rad/step is still reachable,
   so the parity gate still passes.
5. token_saliency: F.pad(beta, (0,0,0,0,0,1)) padded the BATCH dim of a 3-D tensor
   (beta is (B,L,H)), so beta_next came out length L-1 -> the
   "size of tensor a (16383) must match tensor b (16384)" crash in the XAI cell.
6. _reverse_energy was a Python loop over L (16384 steps x 4 directions x 8 blocks).
   It is now a vectorised reverse scan reusing the same bounded machinery.

MIMO extension (not enabled): make B_proj / C_proj emit (N * r) and reshape to
(N, r), replace the outer product with a matmul over the rank-r channel group.
Enable only once validated -- until then, SISO is what runs, and that is what any
paper text should claim.
"""
from __future__ import annotations

import math
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

_NEG_INF = float("-inf")


def _rope(v: Tensor, phi: Tensor) -> Tensor:
    """Rotate the state dimension of v in 2-D blocks by angles phi.

    v:   (..., N)      N even; consecutive pairs form the 2-D rotation blocks.
    phi: (..., N // 2) rotation angle per block.
    """
    v2 = v.unflatten(-1, (-1, 2))                      # (..., N/2, 2)
    cos, sin = torch.cos(phi), torch.sin(phi)
    x0, x1 = v2[..., 0], v2[..., 1]
    return torch.stack((x0 * cos - x1 * sin, x0 * sin + x1 * cos), dim=-1).flatten(-2)


def _causal_decay(cs: Tensor) -> Tensor:
    """D[b,c,t,s,h] = exp(cs_t - cs_s) for s <= t, else 0.   cs: (B, nc, chunk, H).

    cs is an inclusive cumsum of log(alpha) <= 0, so for s <= t the exponent is <= 0
    and D is in [0, 1]. The strictly-upper triangle would have a positive exponent,
    so it is masked to -inf BEFORE the exp (never after -- that would overflow).
    """
    c = cs.shape[2]
    dec = cs.unsqueeze(3) - cs.unsqueeze(2)            # (B, nc, t, s, H)
    causal = torch.ones(c, c, dtype=torch.bool, device=cs.device).tril()
    return torch.exp(dec.masked_fill(~causal[None, None, :, :, None], _NEG_INF))


class Mamba3RefSSM(nn.Module):
    """Selective SSM with exponential-trapezoidal discretization + complex (RoPE) states.

    Input/output are (B, L, D).
    """

    def __init__(
        self,
        d_model: int,
        d_state: int = 64,
        expand: int = 2,
        headdim: int = 64,
        dt_min: float = 1e-3,
        dt_max: float = 1e-1,
        rope_base: float = 10000.0,
        use_rope: bool = True,
        trapezoidal: bool = True,
        chunk: int = 64,
        rot_max: float = 2.0 * math.pi,
    ):
        super().__init__()
        assert d_state % 2 == 0, "d_state must be even for 2-D rotary blocks"
        self.use_rope = use_rope          # False -> real-valued diagonal SSM (ablation control)
        self.trapezoidal = trapezoidal    # False -> lambda=1 = exponential-Euler = Mamba-2
        self.parallel = True              # chunked SSD scan; False -> O(L) reference loop
        self.chunk = int(chunk)
        self.rot_max = float(rot_max)     # max |rotation rate| per step (>= pi -> parity reachable)
        self.d_model = d_model
        self.d_inner = int(expand * d_model)
        self.headdim = min(headdim, self.d_inner)
        assert self.d_inner % self.headdim == 0, "d_inner must be divisible by headdim"
        self.nheads = self.d_inner // self.headdim
        self.d_state = d_state

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)   # x, z(gate)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.dt_proj = nn.Linear(self.d_inner, self.nheads, bias=True)
        self.B_proj = nn.Linear(self.d_inner, d_state, bias=True)
        self.C_proj = nn.Linear(self.d_inner, d_state, bias=True)
        self.lam_proj = nn.Linear(self.d_inner, self.nheads, bias=True)
        # Rotation RATE (imaginary part of complex A): data-dependent, decoupled from the
        # small real decay step dt, and bounded so cumulative angles stay well conditioned.
        # Must reach ~pi/step to represent parity-style rotational state tracking.
        self.w_proj = nn.Linear(self.d_inner, self.nheads, bias=True)

        # A < 0 via -exp(A_log); ONE scalar decay rate per head (Mamba-2/3 SSD structure).
        self.A_log = nn.Parameter(torch.zeros(self.nheads))
        self.norm = nn.LayerNorm(self.d_inner)

        theta = rope_base ** (-torch.arange(0, d_state, 2).float() / d_state)  # (N/2,)
        self.register_buffer("theta", theta, persistent=False)

        with torch.no_grad():
            dt = torch.empty(self.nheads).uniform_(math.log(dt_min), math.log(dt_max)).exp()
            self.dt_proj.bias.copy_(dt + torch.log(-torch.expm1(-dt)))  # inverse softplus

    # ------------------------------------------------------------------ pieces
    def _rotate(self, uf: Tensor, Bt: Tensor, Ct: Tensor) -> Tuple[Tensor, Tensor]:
        B, L, _ = uf.shape
        H, N = self.nheads, self.d_state
        Bh = Bt.unsqueeze(2).expand(B, L, H, N)
        Ch = Ct.unsqueeze(2).expand(B, L, H, N)
        if not self.use_rope:
            return Bh, Ch                              # real-valued control: no rotation
        w = self.rot_max * torch.tanh(self.w_proj(uf))          # (B,L,H) in (-rot_max, rot_max)
        phi = torch.cumsum(w.unsqueeze(-1) * self.theta, dim=1)  # (B,L,H,N/2)
        phi = torch.remainder(phi, 2.0 * math.pi)                # exact, keeps cos/sin accurate
        return _rope(Bh, phi), _rope(Ch, phi)

    def _coeffs(self, uf: Tensor):
        """dt/lambda/A -> (log_alpha, gamma, beta). beta is None for the Euler ablation."""
        dt = F.softplus(self.dt_proj(uf))                        # (B,L,H) > 0
        log_alpha = dt * (-torch.exp(self.A_log.float()))        # (B,L,H) <= 0
        if not self.trapezoidal:
            return log_alpha, dt, None                           # lambda = 1
        lam = torch.sigmoid(self.lam_proj(uf))                   # (B,L,H) in [0,1]
        gamma = lam * dt
        beta = (1.0 - lam) * dt * torch.exp(log_alpha)
        return log_alpha, gamma, beta

    # ------------------------------------------------------------------ forward
    def forward(self, x: Tensor) -> Tensor:
        B, L, _ = x.shape
        H, P = self.nheads, self.headdim

        xz = self.in_proj(x)
        u, z = xz.chunk(2, dim=-1)                     # (B, L, d_inner) each
        u = self.norm(u)

        # The SSM core is not fp16-safe (exp / cumsum / rotation over thousands of
        # tokens). The projections above stay in AMP; only the core is forced to fp32.
        with torch.autocast(device_type=x.device.type, enabled=False):
            uf = u.float()
            log_alpha, gamma, beta = self._coeffs(uf)
            Brot, Crot = self._rotate(uf, self.B_proj(uf), self.C_proj(uf))
            uh = uf.view(B, L, H, P)
            scan = self._ssd if self.parallel else self._ssm_reference
            y = scan(log_alpha, gamma, beta, uh, Brot, Crot)     # (B,L,H,P)
            y = y.reshape(B, L, self.d_inner)

        y = y.to(z.dtype) * F.silu(z)                  # gate
        return self.out_proj(y)

    # ------------------------------------------------------------------ scans
    @staticmethod
    def _shift(t: Tensor) -> Tensor:
        """x_{t-1} along dim 1, zero-filled at t=0. t: (B, L, ...) with 2 trailing dims."""
        L = t.shape[1]
        return F.pad(t, (0, 0, 0, 0, 1, 0))[:, :L]

    def _ssd(self, log_alpha: Tensor, gamma: Tensor, beta: Optional[Tensor],
             x: Tensor, Br: Tensor, Cr: Tensor) -> Tensor:
        """Chunked segment-sum scan. Mathematically identical to _ssm_reference.

        log_alpha, gamma, beta: (B,L,H);  x: (B,L,H,P);  Br, Cr: (B,L,H,N)  -> y: (B,L,H,P)

        Because A is scalar per head, the readout <h_t, C_t> can be contracted in closed
        form, so the (B,L,H,P,N) state trajectory is never built:
            y_t = sum_{s<=t} D[t,s] * (B_s . C_t) * w_s * x_s        (intra-chunk)
                + exp(cs_t) * <h_chunk_start, C_t>                   (inter-chunk)
        """
        B, L, H, P = x.shape
        N = Br.shape[-1]
        c = max(1, min(self.chunk, L))
        xp, Bp = self._shift(x), self._shift(Br)       # trapezoidal (previous-token) drive

        pad = (-L) % c
        if pad:
            log_alpha = F.pad(log_alpha, (0, 0, 0, pad))          # log alpha = 0 -> alpha = 1
            gamma = F.pad(gamma, (0, 0, 0, pad))                  # gamma = 0 -> no drive
            beta = None if beta is None else F.pad(beta, (0, 0, 0, pad))
            x, xp = F.pad(x, (0, 0, 0, 0, 0, pad)), F.pad(xp, (0, 0, 0, 0, 0, pad))
            Br, Bp = F.pad(Br, (0, 0, 0, 0, 0, pad)), F.pad(Bp, (0, 0, 0, 0, 0, pad))
            Cr = F.pad(Cr, (0, 0, 0, 0, 0, pad))
        Lp, nc = L + pad, (L + pad) // c

        la = log_alpha.view(B, nc, c, H)
        gc = gamma.view(B, nc, c, H)
        xc, xpc = x.view(B, nc, c, H, P), xp.view(B, nc, c, H, P)
        Brc, Bpc = Br.view(B, nc, c, H, N), Bp.view(B, nc, c, H, N)
        Crc = Cr.view(B, nc, c, H, N)
        bc = None if beta is None else beta.view(B, nc, c, H)

        cs = la.cumsum(2)                              # (B,nc,c,H) inclusive
        D = _causal_decay(cs)                          # (B,nc,t,s,H) in [0,1]

        # ---- intra-chunk readout
        att = torch.einsum("bcthn,bcshn->bctsh", Crc, Brc) * D * gc.unsqueeze(2)
        y = torch.einsum("bctsh,bcshp->bcthp", att, xc)
        if bc is not None:
            attp = torch.einsum("bcthn,bcshn->bctsh", Crc, Bpc) * D * bc.unsqueeze(2)
            y = y + torch.einsum("bctsh,bcshp->bcthp", attp, xpc)

        # ---- per-chunk end state, then the only sequential part: nc = L/chunk steps
        Sc = torch.exp(cs[:, :, -1:, :] - cs)          # (B,nc,c,H) in (0,1]: decay s -> chunk end
        h_local = torch.einsum("bcshp,bcshn->bchpn", (Sc * gc).unsqueeze(-1) * xc, Brc)
        if bc is not None:
            h_local = h_local + torch.einsum("bcshp,bcshn->bchpn",
                                             (Sc * bc).unsqueeze(-1) * xpc, Bpc)
        adv = torch.exp(cs[:, :, -1, :])               # (B,nc,H) full-chunk decay, in (0,1]
        states, h = [], x.new_zeros(B, H, P, N)
        for i in range(nc):
            states.append(h)                           # state entering chunk i
            h = adv[:, i, :, None, None] * h + h_local[:, i]
        states = torch.stack(states, 1)                # (B,nc,H,P,N)

        # ---- inter-chunk readout
        y = y + torch.einsum("bcthn,bchpn->bcthp", Crc, states) * torch.exp(cs).unsqueeze(-1)
        return y.reshape(B, Lp, H, P)[:, :L]

    def _ssm_reference(self, log_alpha: Tensor, gamma: Tensor, beta: Optional[Tensor],
                       x: Tensor, Br: Tensor, Cr: Tensor) -> Tensor:
        """Sequential O(L) ground truth. Correct and honest, not fast."""
        B, L, H, P = x.shape
        N = Br.shape[-1]
        alpha = torch.exp(log_alpha)
        xp, Bp = self._shift(x), self._shift(Br)
        h = x.new_zeros(B, H, P, N)
        out = []
        for t in range(L):
            U = gamma[:, t, :, None, None] * x[:, t, :, :, None] * Br[:, t, :, None, :]
            if beta is not None:
                U = U + beta[:, t, :, None, None] * xp[:, t, :, :, None] * Bp[:, t, :, None, :]
            h = alpha[:, t, :, None, None] * h + U
            out.append((h * Cr[:, t, :, None, :]).sum(-1))
        return torch.stack(out, dim=1)

    def _scan_scalar(self, log_a: Tensor, u: Tensor) -> Tensor:
        """h_j = exp(log_a_j) * h_{j-1} + u_j, vectorised. Both (B, L, H) -> (B, L, H)."""
        B, L, H = u.shape
        c = max(1, min(self.chunk, L))
        pad = (-L) % c
        if pad:
            log_a, u = F.pad(log_a, (0, 0, 0, pad)), F.pad(u, (0, 0, 0, pad))
        Lp, nc = L + pad, (L + pad) // c
        la, uc = log_a.view(B, nc, c, H), u.view(B, nc, c, H)
        cs = la.cumsum(2)
        y = torch.einsum("bctsh,bcsh->bcth", _causal_decay(cs), uc)
        Sc = torch.exp(cs[:, :, -1:, :] - cs)
        h_local = (Sc * uc).sum(2)                     # (B,nc,H)
        adv = torch.exp(cs[:, :, -1, :])
        states, h = [], u.new_zeros(B, H)
        for i in range(nc):
            states.append(h)
            h = adv[:, i] * h + h_local[:, i]
        y = y + torch.stack(states, 1).unsqueeze(2) * torch.exp(cs)
        return y.reshape(B, Lp, H)[:, :L]

    def _reverse_energy(self, log_alpha: Tensor) -> Tensor:
        """G_tau = sum_{t>=tau} (prod_{k=tau+1..t} alpha_k)^2, i.e. the stable reverse
        recurrence G_tau = 1 + alpha_{tau+1}^2 * G_{tau+1}, G_{L-1} = 1.  (B,L,H).

        Written as a forward scan on the reversed sequence so it costs O(L/chunk) steps
        instead of O(L) Python iterations (L is 4k-16k tokens per block here).
        """
        L = log_alpha.shape[1]
        af = torch.flip(2.0 * log_alpha, dims=[1])               # af[j] = 2*log alpha_{L-1-j}
        logc = F.pad(af, (0, 0, 1, 0))[:, :L]                    # c_j = alpha_{L-j}^2, c_0 free
        g = self._scan_scalar(logc, torch.ones_like(logc))       # g_j = c_j g_{j-1} + 1
        return torch.flip(g, dims=[1])

    # ------------------------------------------------------------------ XAI
    @torch.no_grad()
    def token_saliency(self, x: Tensor) -> Tensor:
        """Intrinsic controllability-energy saliency per input token. Returns (B, L).
        Closed-form from the SSM internals, one pass, no backprop.

        Accounts for BOTH Gramian contributions of each token:
          - gamma term: token t's direct drive at step t
          - beta  term: token t's carry-forward drive at step t+1 (trapezoidal)
        """
        B, L, _ = x.shape
        H, P = self.nheads, self.headdim
        u, _ = self.in_proj(x).chunk(2, dim=-1)
        u = self.norm(u)
        with torch.autocast(device_type=x.device.type, enabled=False):
            uf = u.float()
            log_alpha, gamma, beta = self._coeffs(uf)
            Bt = self.B_proj(uf)
            Brot, _ = self._rotate(uf, Bt, Bt)
            uh = uf.view(B, L, H, P)
            G = self._reverse_energy(log_alpha)                  # (B,L,H)
            bnorm = (Brot ** 2).sum(-1)                          # (B,L,H)
            xnorm = (uh ** 2).sum(-1)                            # (B,L,H)
            s = (gamma ** 2) * xnorm * bnorm * G                 # token t drives state at t
            if beta is not None:
                # token t also drives the state at t+1 with weight beta_{t+1}
                beta_next = F.pad(beta, (0, 0, 0, 1))[:, 1:L + 1]
                G_next = F.pad(G, (0, 0, 0, 1))[:, 1:L + 1]
                s = s + (beta_next ** 2) * xnorm * bnorm * G_next
        return s.mean(-1)                                        # (B, L)

In [ ]:
%%writefile src/blocks/mamba3_odss.py
"""
Mamba3ODSSBlock -- omni-directional selective scan block built on the verified
pure-PyTorch Mamba-3 core (src/blocks/mamba3_ref.Mamba3RefSSM).

Change vs. the first version: the official-kernel path no longer silently doubles the
parameter count.  `self._official_mamba = Mamba3(...)` DID register as a submodule
(nn.Module.__setattr__ registers any nn.Module value regardless of the leading
underscore), so the reported params were the reference + the kernel.  Exactly one of
the two paths is now constructed, so `params` means what it says.  With no released
Mamba-3 kernel (HAS_MAMBA3 is False on Kaggle) the reference is what runs -- keep
reporting official_mamba3_kernels = False.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.utils.checkpoint as _ckpt
from torch import Tensor

try:
    from mamba_ssm.modules.mamba3 import Mamba3
    HAS_MAMBA3 = True
except Exception:
    Mamba3 = None
    HAS_MAMBA3 = False

from src.blocks.mamba3_ref import Mamba3RefSSM


class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: Tensor) -> Tensor:
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = x.new_empty(shape).bernoulli_(keep)
        return x * mask / keep


class LSBlock(nn.Module):
    def __init__(self, dim: int, kernel_size: int = 3):
        super().__init__()
        self.dw = nn.Conv2d(dim, dim, kernel_size, padding=kernel_size // 2, groups=dim, bias=False)
        self.bn = nn.BatchNorm2d(dim)
        self.pw = nn.Conv2d(dim, dim, 1, bias=False)
        self.act = nn.GELU()

    def forward(self, x: Tensor) -> Tensor:
        residual = x
        x = self.pw(self.act(self.bn(self.dw(x))))
        return x + residual


class RGBlock(nn.Module):
    def __init__(self, dim: int, expansion: float = 2.0):
        super().__init__()
        hidden = int(dim * expansion)
        self.fc1 = nn.Conv2d(dim, hidden, 1, bias=False)
        self.fc2 = nn.Conv2d(dim, hidden, 1, bias=False)
        self.dw = nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden, bias=False)
        self.fc_out = nn.Conv2d(hidden, dim, 1, bias=False)
        self.act = nn.SiLU()
        self.norm = nn.BatchNorm2d(dim)

    def forward(self, x: Tensor) -> Tensor:
        residual = x
        x = self.act(self.fc1(x)) * self.fc2(x)
        x = self.fc_out(self.dw(x))
        return self.norm(x + residual)


class Mamba3SS2D(nn.Module):
    """4-directional selective scan over a feature map, driven by the Mamba-3 SSM."""

    def __init__(self, dim: int, d_state: int = 64, expand: int = 2, headdim: int = 64,
                 is_mimo: bool = True, mimo_rank: int = 4, drop_path: float = 0.0,
                 use_rope: bool = True, trapezoidal: bool = True):
        super().__init__()
        self.dim = dim
        self.norm = nn.LayerNorm(dim)
        self.drop_path = DropPath(drop_path) if drop_path > 0 else nn.Identity()
        # Exactly ONE scan implementation is constructed -> params are not double counted.
        self.official = None
        self.ref = None
        if HAS_MAMBA3:
            try:
                self.official = Mamba3(d_model=dim, d_state=min(d_state, 64), expand=expand,
                                       headdim=min(headdim, 64), is_mimo=False, mimo_rank=1)
            except Exception:
                self.official = None
        if self.official is None:
            # Real Mamba-3 SSM reference (verified on the parity gate + the O(L) equivalence
            # gate). d_state forced even for the 2-D rotary blocks. use_rope / trapezoidal
            # are the mechanism-ablation toggles.
            self.ref = Mamba3RefSSM(dim, d_state=d_state + (d_state % 2), expand=expand,
                                    headdim=headdim, use_rope=use_rope, trapezoidal=trapezoidal)

    @property
    def uses_official_kernel(self) -> bool:
        return self.official is not None

    def _scan(self, seq: Tensor) -> Tensor:
        seq = seq.contiguous()
        return self.official(seq) if self.official is not None else self.ref(seq)

    def _four_dir_scan(self, x: Tensor) -> Tensor:
        B, C, H, W = x.shape
        x = x.contiguous()
        seq_h = x.flatten(2).transpose(1, 2).contiguous()
        seq_hf = torch.flip(seq_h, dims=[1]).contiguous()
        seq_v = x.permute(0, 1, 3, 2).contiguous().flatten(2).transpose(1, 2).contiguous()
        seq_vf = torch.flip(seq_v, dims=[1]).contiguous()
        out_h = self._scan(seq_h).transpose(1, 2).contiguous().view(B, C, H, W)
        out_hf = torch.flip(self._scan(seq_hf), dims=[1]).transpose(1, 2).contiguous().view(B, C, H, W)
        out_v = self._scan(seq_v).transpose(1, 2).contiguous().view(B, C, W, H).permute(0, 1, 3, 2).contiguous()
        out_vf = torch.flip(self._scan(seq_vf), dims=[1]).transpose(1, 2).contiguous() \
                      .view(B, C, W, H).permute(0, 1, 3, 2).contiguous()
        return (out_h + out_hf + out_v + out_vf) * 0.25

    def forward(self, x: Tensor) -> Tensor:
        residual = x
        x = x.permute(0, 2, 3, 1).contiguous()
        x = self.norm(x)
        x = x.permute(0, 3, 1, 2).contiguous()
        x = self._four_dir_scan(x)
        return residual + self.drop_path(x)

    @torch.no_grad()
    def spatial_saliency(self, x: Tensor) -> Tensor:
        """Intrinsic controllability-Gramian saliency map for this block. (B,H,W).
        Runs the 4 scan directions and folds each token-saliency back to 2-D."""
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1).contiguous()
        x = self.norm(x)
        x = x.permute(0, 3, 1, 2).contiguous()
        sal = getattr(self.ref, "token_saliency", None) if self.ref is not None else None
        if sal is None:      # official-kernel path exposes no intrinsic saliency
            return x.new_zeros(B, H, W)
        seq_h = x.flatten(2).transpose(1, 2).contiguous()
        seq_hf = torch.flip(seq_h, dims=[1]).contiguous()
        seq_v = x.permute(0, 1, 3, 2).contiguous().flatten(2).transpose(1, 2).contiguous()
        seq_vf = torch.flip(seq_v, dims=[1]).contiguous()
        mh = sal(seq_h).view(B, H, W)
        mhf = torch.flip(sal(seq_hf), dims=[1]).view(B, H, W)
        mv = sal(seq_v).view(B, W, H).permute(0, 2, 1)
        mvf = torch.flip(sal(seq_vf), dims=[1]).view(B, W, H).permute(0, 2, 1)
        return (mh + mhf + mv + mvf) * 0.25


class Mamba3ODSSBlock(nn.Module):
    def __init__(self, dim: int, d_state: int = 64, expand: int = 2, headdim: int = 64,
                 is_mimo: bool = True, mimo_rank: int = 4, drop_path: float = 0.0,
                 mlp_ratio: float = 2.0, use_rope: bool = True, trapezoidal: bool = True):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(dim, dim, 1, bias=False), nn.BatchNorm2d(dim),
                                   nn.SiLU(inplace=True))
        self.ls = LSBlock(dim)
        self.ss2d = Mamba3SS2D(dim=dim, d_state=d_state, expand=expand, headdim=headdim,
                               is_mimo=is_mimo, mimo_rank=mimo_rank, drop_path=drop_path,
                               use_rope=use_rope, trapezoidal=trapezoidal)
        self.rg = RGBlock(dim, expansion=mlp_ratio)
        self.use_checkpoint = True   # recompute the SSM scan in backward -> memory saving

    def forward(self, x: Tensor) -> Tensor:
        x = self.ls(self.conv1(x))
        if self.use_checkpoint and self.training and x.requires_grad:
            x = _ckpt.checkpoint(self.ss2d, x, use_reentrant=False)
        else:
            x = self.ss2d(x)
        return self.rg(x)


def build_mamba3_odss(c1: int, c2: int, n: int = 1, **kwargs) -> nn.Module:
    if c1 != c2:
        return nn.Sequential(nn.Conv2d(c1, c2, 1, bias=False), nn.BatchNorm2d(c2),
                             nn.SiLU(inplace=True),
                             *[Mamba3ODSSBlock(c2, **kwargs) for _ in range(n)])
    return nn.Sequential(*[Mamba3ODSSBlock(c2, **kwargs) for _ in range(n)])


class Mamba3ODSS(nn.Module):
    """Ultralytics-compatible wrapper (c1, c2, n, ...) so a YOLO yaml can reference
    `Mamba3ODSS` where it would use C3k2. Matches the channel+repeat calling convention
    Ultralytics' parse_model uses for CSP blocks. Leaner defaults (expand=1,
    mlp_ratio=1.0, d_state=32) keep the -s variant near ~12-13M."""

    def __init__(self, c1: int, c2: int, n: int = 1, d_state: int = 32,
                 expand: int = 1, mlp_ratio: float = 1.0,
                 use_rope: bool = True, trapezoidal: bool = True):
        super().__init__()
        self.block = build_mamba3_odss(c1, c2, n=max(int(n), 1), d_state=d_state,
                                       expand=expand, mlp_ratio=mlp_ratio,
                                       use_rope=bool(use_rope), trapezoidal=bool(trapezoidal))

    def forward(self, x: Tensor) -> Tensor:
        return self.block(x)

In [ ]:
%%writefile scripts/ultra_mamba3.py
#!/usr/bin/env python3
"""
Register Mamba3ODSS with Ultralytics so a YOLO yaml can use it, and generate a
yolo11-mamba3 config (C3k2 mixers -> Mamba3ODSS). This makes the Mamba-3 block
survive Ultralytics' yaml rebuild during .train() -- the surgery-on-object route
does NOT (train() re-parses from cfg).

    from scripts.ultra_mamba3 import register, make_yaml
    register()
    from ultralytics import YOLO
    model = YOLO(make_yaml("s"))          # yolo11s with Mamba-3 mixers
    model.train(data="coco.yaml", ...)    # real DFL loss + assigner + NMS + mAP

No site-packages files are modified: parse_model is re-bound in-process.

ATTRIBUTION: the ODSS / LSBlock / RGBlock design this plugs into is Mamba-YOLO's
(Wang et al., AAAI 2025, arXiv:2406.05835, https://github.com/HZAI-ZJNU/Mamba-YOLO,
AGPL-3.0). The C3k2 -> SSM-mixer swap on a YOLO11 yaml is also not new: see
Xray-YOLO-Mamba (Zhao et al., Sci Rep 15:13171, 2025), which replaces C3k2 with a
VSS/SS2D block in the YOLOv11-n backbone. What is new here is the SSM *inside* the
block (Mamba-3: complex/rotational state + exponential-trapezoidal discretization)
and the intrinsic Gramian saliency. Say exactly that and nothing more.

`where="backbone"` exists because no prior work swaps the neck: Xray-YOLO-Mamba
keeps the YOLOv11 neck, Mamba-YOLO tunes a stage ratio. Swapping all 8 mixers is
what takes this model to ~69 GFLOPs vs Mamba-YOLO-T's 13.2G. Measure both.
"""
from __future__ import annotations

import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))


def register(verbose: bool = True) -> None:
    """Patch ultralytics.nn.tasks.parse_model so it treats Mamba3ODSS as a
    channel-changing, repeatable CSP block (like C3k2)."""
    import ultralytics.nn.tasks as T
    from src.blocks.mamba3_odss import Mamba3ODSS
    if getattr(T, "_mamba3_registered", False):
        return

    src = inspect.getsource(T.parse_model)
    a_rep = "args.insert(2, n)"           # the repeat-insert set sits just before this
    a_chan = "c1, c2 = ch[f], args[0]"    # the channel set sits just before this
    if a_rep not in src or a_chan not in src:
        raise RuntimeError("parse_model anchors not found; ultralytics layout changed "
                           "(pinned 8.3.0). Update scripts/ultra_mamba3.py.")

    def inject(text: str, anchor: str) -> str:
        i = text.index(anchor)
        brace = text.rindex("}", 0, i)    # closing brace of the set literal before anchor
        return text[:brace] + "    Mamba3ODSS,\n            " + text[brace:]

    src = inject(src, a_rep)
    src = inject(src, a_chan)

    T.Mamba3ODSS = Mamba3ODSS                            # visible to the set literal at call time
    exec(compile(src, T.__file__, "exec"), T.__dict__)   # re-bind parse_model in-place
    T._mamba3_registered = True
    if verbose:
        print("registered Mamba3ODSS with ultralytics parse_model")


def make_yaml(scale: str = "s", d_state: int = 32, expand: int = 1,
              mlp_ratio: float = 1.0, use_rope: bool = True, trapezoidal: bool = True,
              where: str = "all", tag: str | None = None, out: str | None = None) -> str:
    """Generate a yolo11-mamba3 yaml from the stock yolo11 config, swapping C3k2 mixers
    for Mamba3ODSS. Ablation flags are baked into each block's args:
    [c2, d_state, expand, mlp_ratio, use_rope, trapezoidal].

    where: "all"      -> swap backbone + neck (8 mixers; ~69 GFLOPs at -s)
           "backbone" -> swap backbone only, keep the stock YOLO11 neck (what every
                         prior Mamba-YOLO variant does). Cheaper; ablate against "all".
    Returns the path."""
    import ultralytics
    import yaml
    assert where in {"all", "backbone"}, "where must be 'all' or 'backbone'"
    stock = Path(ultralytics.__file__).parent / "cfg/models/11/yolo11.yaml"
    d = yaml.safe_load(stock.read_text())
    extra = [int(d_state), int(expand), float(mlp_ratio), bool(use_rope), bool(trapezoidal)]

    def swap(layers):
        for layer in layers:                       # layer = [from, number, module, args]
            if layer[2] == "C3k2":
                layer[2] = "Mamba3ODSS"
                layer[3] = [layer[3][0], *extra]   # out-channels + ablation args
        return layers

    d["backbone"] = swap(d["backbone"])
    if where == "all":
        d["head"] = swap(d["head"])

    if tag is None:
        tag = ""
        if where == "backbone":
            tag += "-bb"
        if not use_rope:
            tag += "-norope"
        if not trapezoidal:
            tag += "-euler"
    out_dir = Path(__file__).resolve().parents[1] / "configs/models"
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out or str(out_dir / f"yolo11{scale}-mamba3{tag}.yaml")
    Path(out).write_text(yaml.safe_dump(d, sort_keys=False))
    return out


if __name__ == "__main__":
    import torch
    from ultralytics import YOLO
    from src.blocks.mamba3_ref import Mamba3RefSSM
    register()
    for where in ("all", "backbone"):
        path = make_yaml("s", where=where)
        m = YOLO(path)
        n = sum(isinstance(mm, Mamba3RefSSM) for mm in m.model.modules())
        p = sum(pp.numel() for pp in m.model.parameters()) / 1e6
        print(f"{Path(path).name}: Mamba3RefSSM blocks={n}, params={p:.2f}M")
        with torch.no_grad():
            out = m.model(torch.randn(1, 3, 320, 320))
        print("  forward ok" if out is not None else "  forward FAILED")

In [ ]:
%%writefile src/xai/gramian.py
"""
Intrinsic controllability-Gramian explainability for Mamba3Yolo.

Unlike Grad-CAM (gradient-based, saturates on recurrent nets) this reads the
attribution directly from the SSM's own dynamics -- one forward pass, no backprop,
O(L). It is closed-form precisely because Mamba-3's state is complex-diagonal.

  from src.xai.gramian import gramian_saliency
  heat = gramian_saliency(model, img)     # (B, H, W) in [0,1], per input pixel

This is the paper's novel XAI contribution. Compare against Grad-CAM++ (src/xai/gradcam.py)
on insertion/deletion + pointing-game -- the numbers competitors (e.g. AKCMamba-YOLO,
which only shows qualitative Grad-CAM) never report.
"""
from __future__ import annotations

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from typing import Optional

import torch
import torch.nn.functional as F
from torch import Tensor

from src.blocks.mamba3_odss import Mamba3SS2D


def _norm(t: Tensor) -> Tensor:
    B = t.shape[0]
    flat = t.reshape(B, -1)
    lo = flat.min(1, keepdim=True).values
    hi = flat.max(1, keepdim=True).values
    return ((flat - lo) / (hi - lo + 1e-8)).view_as(t)


@torch.no_grad()
def gramian_saliency(model: torch.nn.Module, x: Tensor,
                     size: Optional[tuple] = None) -> Tensor:
    """Aggregate intrinsic saliency over every Mamba3SS2D block. Returns (B, H, W)."""
    blocks = [m for m in model.modules() if isinstance(m, Mamba3SS2D)]
    if not blocks:
        raise ValueError("no Mamba3SS2D blocks in model")
    if all(b.ref is None for b in blocks):
        raise ValueError("intrinsic saliency needs the pure-PyTorch reference core "
                         "(the official kernel exposes no internals)")
    captured: dict = {}
    hooks = [b.register_forward_pre_hook(
        lambda mod, inp, key=b: captured.__setitem__(key, inp[0].detach())) for b in blocks]
    was_training = model.training
    model.eval()
    try:
        model(x)
    finally:
        for h in hooks:
            h.remove()
        if was_training:
            model.train()

    H0, W0 = size or x.shape[-2:]
    total = None
    for b, inp in captured.items():
        sal = b.spatial_saliency(inp)                      # (B,h,w)
        sal = F.interpolate(sal.unsqueeze(1).float(), size=(H0, W0),
                            mode="bilinear", align_corners=False).squeeze(1)
        sal = _norm(sal)
        total = sal if total is None else total + sal
    return _norm(total)


# ---------------------------------------------------------------- self-tests
if __name__ == "__main__":
    from src.blocks.mamba3_ref import Mamba3RefSSM

    torch.manual_seed(0)
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    # 1) reverse-energy G matches brute force
    ssm = Mamba3RefSSM(d_model=24, d_state=8, headdim=12).to(dev).eval()
    B, L, H = 2, 40, ssm.nheads
    alpha = (torch.rand(B, L, H, device=dev) * 0.3 + 0.6)      # decays in [0.6, 0.9]
    G_fast = ssm._reverse_energy(alpha.log())
    G_brute = torch.zeros_like(alpha)
    for tau in range(L):                                       # G_tau = sum_{t>=tau} (prod alpha)^2
        D = torch.ones(B, H, device=dev)
        for t in range(tau, L):
            if t > tau:
                D = D * alpha[:, t]
            G_brute[:, tau] += D ** 2
    err = (G_fast - G_brute).abs().max().item()
    print(f"[reverse-energy] max|fast-brute| = {err:.2e}  ->", "OK" if err < 1e-3 else "MISMATCH")

    # 2) does the block's saliency localize a bright region?
    ss2d = Mamba3SS2D(dim=16, d_state=8, headdim=16).to(dev).eval()
    feat = torch.randn(1, 16, 16, 16, device=dev) * 0.1
    feat[:, :, 3:7, 10:14] += 3.0                              # bright patch top-right
    sal = ss2d.spatial_saliency(feat)[0]
    patch = sal[3:7, 10:14].mean().item()
    elsewhere = ((sal.sum() - sal[3:7, 10:14].sum()) / (sal.numel() - 16)).item()
    print(f"[block saliency] patch={patch:.3f} vs elsewhere={elsewhere:.3f}  ->",
          "LOCALIZES" if patch > 2 * elsewhere else "weak")

    # 3) end-to-end on the integrated YOLO11-mamba3
    try:
        from scripts.ultra_mamba3 import register, make_yaml
        from ultralytics import YOLO
        register(verbose=False)
        model = YOLO(make_yaml("s")).model.to(dev)
        img = torch.randn(1, 3, 128, 128, device=dev) * 0.1
        img[:, :, 20:44, 80:104] += 2.0                        # bright object
        heat = gramian_saliency(model, img)                    # (1,128,128)
        obj = heat[0, 20:44, 80:104].mean().item()
        bg = ((heat[0].sum() - heat[0, 20:44, 80:104].sum()) / (heat[0].numel() - 24 * 24)).item()
        print(f"[full model] heat shape {tuple(heat.shape)}, obj={obj:.3f} vs bg={bg:.3f}  ->",
              "OBJECT-FOCUSED" if obj > bg else "diffuse")
    except Exception as e:
        print("[full model] skipped:", type(e).__name__, e)

In [ ]:
%%writefile scripts/validate_core.py
#!/usr/bin/env python3
"""
Correctness + stability gates for Mamba3RefSSM. Run this BEFORE spending GPU hours.

Three gates:
  1. EQUIVALENCE  - the chunked SSD scan must equal the O(L) reference recurrence,
                    for trapezoidal on/off and RoPE on/off. If this fails, every
                    number downstream is meaningless.
  2. STABILITY    - forward + backward under AMP autocast must stay finite at the
                    decay/step sizes training actually reaches. This is the gate that
                    the old `Pin.clamp(min=1e-6)` + `U / Pin` scan failed: alpha**chunk
                    underflows, the clamp bites, and U/1e-6 overflows fp16 -> NaN on
                    every batch (the frozen, all-NaN detector runs in the log).
  3. SHAPES       - token_saliency returns (B, L) for odd/even/long sequences. The old
                    F.pad(beta, (0,0,0,0,0,1)) padded the batch dim of a 3-D tensor and
                    returned L-1 -> the 16383-vs-16384 crash in the XAI cell.
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import torch

from src.blocks.mamba3_ref import Mamba3RefSSM

DEV = "cuda" if torch.cuda.is_available() else "cpu"
OK = True


def _fail(msg):
    global OK
    OK = False
    print("  FAIL:", msg)


def gate_equivalence():
    print("[1] chunked SSD scan  vs  O(L) reference recurrence")
    torch.manual_seed(0)
    for trap in (True, False):
        for rope in (True, False):
            m = Mamba3RefSSM(d_model=32, d_state=16, expand=2, headdim=16,
                             use_rope=rope, trapezoidal=trap, chunk=8).to(DEV).eval()
            # push the decay well past the fp16-underflow regime that used to NaN
            with torch.no_grad():
                m.A_log.fill_(1.5)                      # A = -exp(1.5) ~ -4.5
                m.dt_proj.bias.fill_(0.5)
            x = torch.randn(3, 37, 32, device=DEV)      # L not a multiple of chunk
            with torch.no_grad():
                uf = m.norm(m.in_proj(x).chunk(2, -1)[0]).float()
                la, ga, be = m._coeffs(uf)
                Br, Cr = m._rotate(uf, m.B_proj(uf), m.C_proj(uf))
                uh = uf.view(3, 37, m.nheads, m.headdim)
                y_fast = m._ssd(la, ga, be, uh, Br, Cr)
                y_ref = m._ssm_reference(la, ga, be, uh, Br, Cr)
            err = (y_fast - y_ref).abs().max().item()
            scale = y_ref.abs().max().item() + 1e-12
            tag = f"trapezoidal={trap} rope={rope}"
            print(f"  {tag:34s} max|fast-ref| = {err:.3e}  (rel {err/scale:.2e})")
            if not (err / scale < 1e-4):
                _fail(f"{tag}: chunked scan does not match the reference recurrence")


def gate_stability():
    print("[2] AMP forward/backward stays finite at training-scale decays")
    torch.manual_seed(0)
    amp = DEV == "cuda"
    for A_log, dt_bias, L in [(0.0, 0.0, 512), (2.0, 1.0, 512), (4.0, 3.0, 1024)]:
        m = Mamba3RefSSM(d_model=64, d_state=32, expand=1, headdim=64).to(DEV)
        with torch.no_grad():
            m.A_log.fill_(A_log)                        # alpha = exp(-dt*exp(A_log))
            m.dt_proj.bias.fill_(dt_bias)
            m.w_proj.weight.mul_(50.0)                  # try to blow up the rotary phase
        x = torch.randn(2, L, 64, device=DEV, requires_grad=True)
        with torch.autocast(device_type=DEV, dtype=torch.float16, enabled=amp):
            y = m(x)
        loss = y.float().pow(2).mean()
        loss.backward()
        gmax = max(p.grad.abs().max().item() for p in m.parameters() if p.grad is not None)
        finite = torch.isfinite(y).all().item() and torch.isfinite(loss).item()
        gfinite = all(torch.isfinite(p.grad).all().item()
                      for p in m.parameters() if p.grad is not None)
        alpha = float(torch.exp(-torch.nn.functional.softplus(torch.tensor(dt_bias))
                                * torch.exp(torch.tensor(A_log))))
        print(f"  A_log={A_log:<4} dt_bias={dt_bias:<4} L={L:<5} alpha~{alpha:.2e}  "
              f"y_finite={finite} grad_finite={gfinite} max|grad|={gmax:.2e}")
        if not (finite and gfinite):
            _fail(f"non-finite forward/backward at A_log={A_log}, dt_bias={dt_bias}")


def gate_shapes():
    print("[3] token_saliency shapes")
    m = Mamba3RefSSM(d_model=32, d_state=16, expand=1, headdim=32).to(DEV).eval()
    for L in (1, 7, 64, 4096):
        s = m.token_saliency(torch.randn(2, L, 32, device=DEV))
        ok = tuple(s.shape) == (2, L) and torch.isfinite(s).all().item() and (s >= 0).all().item()
        print(f"  L={L:<6} -> {tuple(s.shape)}  finite&nonneg={ok}")
        if not ok:
            _fail(f"token_saliency wrong at L={L}: got {tuple(s.shape)}")


if __name__ == "__main__":
    print(f"device={DEV}\n")
    gate_equivalence()
    print()
    gate_stability()
    print()
    gate_shapes()
    print()
    print("  PASS: core is correct and AMP-stable." if OK else
          "  FAIL: do not train until the gates above are green.")
    sys.exit(0 if OK else 1)

In [ ]:
%%writefile scripts/validate_parity.py
#!/usr/bin/env python3
"""
Honesty gate for Mamba3RefSSM: the parity / state-tracking test.

Mamba-3 claims complex-valued (rotational) states solve running parity
  y_t = (sum_{i<=t} x_i) mod 2
-- a task real-valued diagonal SSMs provably CANNOT (Grazzi et al., Thm 1: real
eigenvalues can't do rotation).

Decisive result is the CONTRAST, not the absolute:
  - use_rope=True  (complex/rotational)  -> should learn parity  (acc -> ~1.0)
  - use_rope=False (real diagonal SSM)   -> should FAIL           (acc ~ 0.5)
If both pass, the rotation isn't doing the work (wiring bug). If the True model
fails, there is no Mamba-3 state-tracking claim. Either way we learn the truth.

Note on the rotation rate: it is now bounded to rot_max * tanh(w) with
rot_max = 2*pi, so pi rad/step (what parity needs) sits at tanh(w) = 0.5 and is
comfortably reachable, while the cumulative phase over a 16k-token image scan can no
longer run away. If you lower rot_max below pi this gate MUST fail -- that is the
point of the bound being an explicit, documented hyperparameter.

CPU-friendly: small model, short sequences, a few hundred steps.
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import torch
import torch.nn as nn

from src.blocks.mamba3_ref import Mamba3RefSSM


class ParityNet(nn.Module):
    def __init__(self, use_rope: bool, d_model=32, d_state=16, headdim=16):
        super().__init__()
        self.embed = nn.Linear(1, d_model)
        self.ssm = Mamba3RefSSM(d_model, d_state=d_state, headdim=headdim, use_rope=use_rope)
        self.head = nn.Linear(d_model, 1)

    def forward(self, bits):                 # bits: (B, L) in {0,1}
        h = self.embed(bits.unsqueeze(-1))   # (B, L, d_model)
        h = self.ssm(h)
        return self.head(h).squeeze(-1)      # (B, L) per-position logit


def batch(B, L, device):
    bits = torch.randint(0, 2, (B, L), device=device).float()
    target = torch.cumsum(bits, dim=1) % 2   # running parity, per position
    return bits, target


def run(use_rope, steps=800, B=128, L=24, seed=0, device="cpu"):
    torch.manual_seed(seed)
    net = ParityNet(use_rope).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    lossf = nn.BCEWithLogitsLoss()
    for _ in range(steps):
        bits, tgt = batch(B, L, device)
        opt.zero_grad()
        loss = lossf(net(bits), tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
    net.eval()
    with torch.no_grad():
        accs = {}
        for evalL in (L, 2 * L):             # same length + longer (length generalization)
            bits, tgt = batch(512, evalL, device)
            accs[evalL] = ((net(bits) > 0).float() == tgt).float().mean().item()
    return accs


if __name__ == "__main__":
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device={dev}\nrunning parity (cumsum mod 2), per-position accuracy\n")
    on = run(use_rope=True, device=dev)
    off = run(use_rope=False, device=dev)
    L = 24
    print(f"  RoPE ON  (complex): L={L} acc={on[L]:.3f}   L={2*L} acc={on[2*L]:.3f}")
    print(f"  RoPE OFF (real)   : L={L} acc={off[L]:.3f}   L={2*L} acc={off[2*L]:.3f}\n")
    gap = on[L] - off[L]
    if on[L] > 0.9 and off[L] < 0.75:
        print(f"  PASS: complex solves parity, real fails (gap {gap:+.3f}). Mamba-3 state-tracking is REAL.")
    elif on[L] > 0.9 and off[L] > 0.9:
        print(f"  SUSPECT: both solve it (gap {gap:+.3f}). Rotation may not be load-bearing -- inspect wiring.")
    elif on[L] < 0.75:
        print(f"  FAIL: complex model cannot track parity (acc {on[L]:.3f}). Claim #2 (Mamba-3 core) is NOT supported.")
    else:
        print(f"  WEAK: gap {gap:+.3f}. Inconclusive -- tune steps/lr and rerun before trusting.")

In [ ]:
import sys; sys.path.insert(0, os.getcwd())

# Neutralize ALL optional third-party logger callbacks in ultralytics 8.3.0 -- each is a
# version-mismatch landmine on Kaggle (ray: _get_session; wandb: curves_results; etc.).
# We don't need any of them; metrics are read from results.csv.
import importlib as _il
for _m in ("raytune", "wb", "tensorboard", "mlflow", "comet", "dvc", "neptune", "clearml", "hub"):
    try:
        _mod = _il.import_module(f"ultralytics.utils.callbacks.{_m}")
        if getattr(_mod, "callbacks", None):
            _mod.callbacks = {k: (lambda *a, **kw: None) for k in _mod.callbacks}
    except Exception:
        pass
print("disabled optional logger callbacks; patched files in place")

## 2. Gates — run these before spending GPU hours

### 2a. Core correctness + AMP stability

Three checks: the chunked SSD scan must equal the O(L) reference recurrence (trapezoidal ×
RoPE, all four combos); forward + backward under autocast must stay finite at the decay
sizes training actually reaches; `token_saliency` must return `(B, L)`.

If any of these is red, **stop** — nothing downstream means anything.

In [ ]:
!python scripts/validate_core.py

### 2b. Honesty gate — parity / state-tracking

Complex (RoPE) state must solve running parity (~1.0); the real-valued control must fail
(~0.5). This is the proof the Mamba-3 core actually tracks state, and it is the one piece
of evidence no prior YOLO-Mamba work has. The contrast is the result, not the absolute.

In [ ]:
!python scripts/validate_parity.py

## 3. Smoke test — real Ultralytics pipeline on coco8

Confirms the model trains through the genuine DFL loss + assigner + NMS + COCO mAP val, and survives Ultralytics' yaml rebuild (surgery-on-object does not).

In [ ]:
from scripts.ultra_mamba3 import register, make_yaml
from ultralytics import YOLO
from src.blocks.mamba3_ref import Mamba3RefSSM
register()
y = YOLO(make_yaml('s'))
print('Mamba blocks before train:', sum(isinstance(m, Mamba3RefSSM) for m in y.model.modules()))
y.train(data='coco8.yaml', epochs=3, imgsz=256, batch=2, device=0, workers=2,
        plots=False, verbose=False, exist_ok=True, name='coco8_smoke')
print('Mamba blocks after train :', sum(isinstance(m, Mamba3RefSSM) for m in y.model.modules()))

## 4. Main comparison + ablations

Same base (YOLO11-s), same data/schedule — only the mixer differs.

**`nbs=BATCH` is not cosmetic.** Ultralytics accumulates gradients to a nominal batch of
`nbs` (default 64). At `batch=2` that is `accumulate=32`, i.e. **two optimizer steps per
epoch** on coco128 — 200 steps in 100 epochs. That, not the architecture, is why the old
baseline stalled at mAP50 0.083 while overfitting 128 images it also validated on.
`accumulate=1` makes an epoch an epoch.

Optimizer is forced (`AdamW`, `lr0=2e-3`) because `optimizer=auto` picks ~1e-4 at small
batch and nothing learns — baseline included. These train from scratch (no pretrained
Mamba weights), which is also what Mamba-YOLO and Xray-YOLO-Mamba do.

In [ ]:
DATA   = 'coco128.yaml'   # <-- a real data.yaml for real numbers (COCO / CLCXray / medical)
EPOCHS = 100
IMGSZ  = 256              # keep BOTH models here for a fair compare; 640 for anything publishable
BATCH  = 8                # the SSD scan removed the (B,L,H,P,N) ceiling -- probe below before raising
DEV    = 0

TKW = dict(optimizer='AdamW', lr0=2e-3, nbs=BATCH, warmup_epochs=5, cos_lr=True,
           plots=False, exist_ok=True)

**Memory probe.** The old scan materialised the full `(B,L,H,P,N)` state trajectory (~33M elements for the stride-4 block alone, ×4 directions); the SSD scan never does, so the ceiling that forced `batch=2, imgsz=256` is gone. Don't take that on faith — measure it, leave ~30% headroom for the val loop and mosaic, and drop `BATCH` or pass `d_state=16` to `make_yaml` if it's tight.

In [ ]:
import torch
from scripts.ultra_mamba3 import register, make_yaml
from ultralytics import YOLO
register()
m = YOLO(make_yaml('s')).model.cuda().train()
torch.cuda.reset_peak_memory_stats()
x = torch.randn(BATCH, 3, IMGSZ, IMGSZ, device='cuda', requires_grad=True)
with torch.autocast('cuda', dtype=torch.float16):
    out = m(x)
sum(o.float().pow(2).mean() for o in (out if isinstance(out, (list, tuple)) else [out])).backward()
print(f'peak {torch.cuda.max_memory_allocated()/2**30:.2f} GiB at batch={BATCH}, imgsz={IMGSZ}')
del m, x, out; torch.cuda.empty_cache()

### 4a. Baseline — stock YOLO11-s (C3k2 mixer)

In [ ]:
from ultralytics import YOLO
YOLO('yolo11s.yaml').train(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEV,
                           name='yolo11s_base', **TKW)

### 4b. Ours + ablations

- `mamba3_full` — RoPE + trapezoidal, all 8 mixers swapped.
- `mamba3_bb` — backbone only, stock YOLO11 neck. **Run this one.** No prior work swaps the
  neck (Xray-YOLO-Mamba keeps the YOLOv11 neck; Mamba-YOLO tunes a stage ratio), and
  swapping all 8 is what takes this model to ~69 GFLOPs against Mamba-YOLO-T's 13.2G.
- `mamba3_norope` — removes the complex state.
- `mamba3_euler` — removes the trapezoidal term (= Mamba-2 discretization).

The last two are the mechanism ablations that carry the whole claim. They are only
interpretable if the full model actually learns something first.

In [ ]:
from scripts.ultra_mamba3 import register, make_yaml
from ultralytics import YOLO
register()
variants = {
    'mamba3_full':   make_yaml('s'),                     # RoPE + trapezoidal, all mixers
    'mamba3_bb':     make_yaml('s', where='backbone'),   # backbone only, stock neck
    'mamba3_norope': make_yaml('s', use_rope=False),     # - complex state
    'mamba3_euler':  make_yaml('s', trapezoidal=False),  # - trapezoidal (= Mamba-2)
}
for name, ycfg in variants.items():
    print('=== training', name, '===')
    YOLO(ycfg).train(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEV,
                     name=name, **TKW)

## 5. Results table

Best epoch, not last. NaN and frozen-run detection is part of the table, not a footnote:
`round(2.7e-05, 4)` printed `0.0000` in the old runbook and looked like a real zero.

In [ ]:
import pandas as pd, os

rows = []
for run in ['yolo11s_base', 'mamba3_full', 'mamba3_bb', 'mamba3_norope', 'mamba3_euler']:
    csv = f'runs/detect/{run}/results.csv'
    if not os.path.exists(csv):
        continue
    df = pd.read_csv(csv); df.columns = [c.strip() for c in df.columns]
    loss_cols = [c for c in df.columns if c.startswith('train/')]
    nan_ep = df.index[df[loss_cols].isna().any(axis=1)]
    best = df.loc[df['metrics/mAP50-95(B)'].idxmax()]
    rows.append({
        'run': run,
        'best_epoch': int(best['epoch']),
        'mAP50': f"{best['metrics/mAP50(B)']:.4g}",
        'mAP50-95': f"{best['metrics/mAP50-95(B)']:.4g}",
        'train_loss_nan': 'NONE' if len(nan_ep) == 0 else f'from ep{int(df.loc[nan_ep[0], "epoch"])}',
        'frozen': bool(df['metrics/mAP50-95(B)'].nunique() <= 2 and len(df) > 10),
    })
print(pd.DataFrame(rows).to_string(index=False) if rows else 'no runs yet')
print('\ntrain_loss_nan must read NONE and frozen must read False for every row.')
print('Anything else means the row is not a result, it is a bug report.')

**Cost table.** Params/FLOPs/latency for every variant, measured here. This is what an efficiency claim has to rest on — not thop's GFLOPs, which only count the Linear/Conv calls and never saw the scan at all.

In [ ]:
import time, torch
from scripts.ultra_mamba3 import register, make_yaml
from ultralytics import YOLO
register()

def cost(cfg, name, imgsz=IMGSZ, n=30):
    m = YOLO(cfg).model.cuda().eval()
    p = sum(q.numel() for q in m.parameters()) / 1e6
    x = torch.randn(1, 3, imgsz, imgsz, device='cuda')
    with torch.no_grad():
        for _ in range(5): m(x)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for _ in range(n): m(x)
        torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / n * 1e3
    mem = torch.cuda.max_memory_allocated() / 2**20
    print(f'{name:16s} params={p:6.2f}M  latency={ms:7.2f} ms  peak={mem:7.0f} MiB  @{imgsz}px')
    del m, x; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

cost('yolo11s.yaml', 'yolo11s')
cost(make_yaml('s'), 'mamba3_full')
cost(make_yaml('s', where='backbone'), 'mamba3_bb')
print('\nReference points (their hardware, their kernel, NOT measured here):')
print('  Mamba-YOLO-T      5.8M / 13.2G / AP 44.5 / 1.5 ms (4090, VMamba CUDA kernel)')
print('  Xray-YOLO-Mamba   4.3M / 10.3G / 95.2 FPS (A100)')
print('  ours: pure-PyTorch scan, no CUDA kernel -- say so before quoting any latency.')

## 6. Gramian explainability on the trained model

Intrinsic controllability-energy saliency — one forward pass, no backprop. Only meaningful on a trained model, and only a contribution if it comes with insertion/deletion + pointing-game numbers vs Grad-CAM++. The priors show heatmaps and assert; don't do that.

In [ ]:
import torch, glob, cv2
import matplotlib.pyplot as plt
from ultralytics.data.augment import LetterBox
from scripts.ultra_mamba3 import register
from ultralytics import YOLO
from src.xai.gramian import gramian_saliency

register()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO('runs/detect/mamba3_full/weights/best.pt').model.to(dev).eval()

imgs = sorted(glob.glob('/kaggle/working/datasets/coco128/images/train2017/*.jpg'))
assert imgs, 'no images found - check the datasets dir from `yolo settings`'
im0 = cv2.cvtColor(cv2.imread(imgs[0]), cv2.COLOR_BGR2RGB)
im = LetterBox((IMGSZ, IMGSZ))(image=im0)          # match training res; 512 makes L=16384 for nothing
x = torch.from_numpy(im).permute(2, 0, 1).float()[None].to(dev) / 255.0

heat = gramian_saliency(model, x)[0].cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(14, 5))
ax[0].imshow(im);                ax[0].set_title('input')
ax[1].imshow(heat, cmap='jet');  ax[1].set_title('Gramian saliency')
ax[2].imshow(im); ax[2].imshow(heat, cmap='jet', alpha=0.5); ax[2].set_title('overlay')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## 7. Integrity checklist

**Gates**
- [ ] `validate_core.py` green: chunked scan == O(L) recurrence, AMP forward/backward finite.
- [ ] `validate_parity.py` green: complex solves parity, real fails. This is the whole
      Mamba-3 claim; without it there is no paper.
- [ ] No run has a NaN in `train/*` of `results.csv`; no run is `frozen`.

**Attribution (do this before the science)**
- [ ] AGPL-3.0 notice in the repo. LSBlock / RGBlock / ODSS credited to Mamba-YOLO
      (AAAI 2025, arXiv:2406.05835). Ultralytics credited.
- [ ] Related work states that the C3k2 → SSM-mixer swap on YOLO11 is Xray-YOLO-Mamba's
      (Sci Rep 15:13171, 2025), not ours.
- [ ] Delete the old fabricated tables (Gemini report / paper_summary.json) and any
      published results from the frozen runs.

**Claims**
- [ ] `official_mamba3_kernels = False`. The verified pure-PyTorch core runs.
- [ ] Efficiency claims are latency + peak memory measured at fixed imgsz/batch, compared
      against Mamba-YOLO-T and Xray-YOLO-Mamba — not thop GFLOPs against YOLO11s.
      thop counts only the Linear/Conv calls and never counted the scan in either version.
- [ ] Ablations use the same schedule as the main model.
- [ ] coco128 numbers are labelled a pipeline check, never a result.

**Before writing anything up**
- [ ] One real dataset at 640px with a comparable schedule (the priors use 600 epochs,
      batch 64, from scratch), with Mamba-YOLO-T and Xray-YOLO-Mamba as rows in the table.
- [ ] `mamba3_bb` vs `mamba3_full` measured — defend 69 GFLOPs or drop the neck swap.
- [ ] XAI faithfulness: insertion/deletion + pointing-game vs Grad-CAM++. See
      `docs/EXPERIMENTS_RUNBOOK.md` (which currently describes the frozen model — fix it).